prepatring for model

In [1]:
import pandas as pd
real_data = pd.read_csv('../datasets/pvAmpSeq_simulation_ready.csv')
real_data.head()

,patient_id,sample_id,episode,marker,allele,frequency,country,MOIs
0,Sero-040,Sero-040-R01,1.0,Chr01,Chr01-1,0.531524,ethiopia,6
1,Sero-182,Sero-182-R12,3.0,Chr11,Chr11-2,0.149320,ethiopia,3
2,Sero-182,Sero-182-R12,3.0,Chr13,Chr13-1,0.729045,ethiopia,3
3,Sero-182,Sero-182-R12,3.0,Chr13,Chr13-2,0.129575,ethiopia,3
4,Sero-182,Sero-182-R12,3.0,Chr14,Chr14-1,0.931269,ethiopia,3


In [2]:
from DeepSets import DataPreprocessor
data_pre = (
    real_data
    .pipe(DataPreprocessor.removing_single_episode_patients)
    .pipe(DataPreprocessor.calculate_population_Allele_frequencies, population='country')
    .pipe(DataPreprocessor.calculate_MOIs)
    .pipe(DataPreprocessor.paired_episode_assigner)
)

In [3]:
import DeepSets

deepest_tensor =DeepSets.build_deepsets_tensors(data_pre)

In [6]:
deepest_tensor['X_alleles'].shape

torch.Size([394, 11, 2, 5, 2])

In [8]:
from torch.utils.data import DataLoader, TensorDataset

# Option 1: TensorDataset
test_dataset = TensorDataset(
    deepest_tensor["X_alleles"],
    deepest_tensor["allele_mask"],
    deepest_tensor["marker_mask"],
    deepest_tensor["priors"],
    deepest_tensor["MOI"]
)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# =====================================================
# Deep Sets model for Pv3Rs surrogate
# =====================================================
class PvDeepSets(nn.Module):
    def __init__(self,
                 n_alleles,        # total number of unique alleles
                 allele_embed_dim=32,
                 marker_embed_dim=64,
                 pair_embed_dim=64,
                 hidden_dim=64,
                 n_classes=3):
        super(PvDeepSets, self).__init__()

        # -------------------------------------------------
        # 1️⃣ Allele embedding (allele ID + frequency)
        # -------------------------------------------------
        self.allele_embedding = nn.Embedding(n_alleles + 1, allele_embed_dim, padding_idx=0)
        self.allele_mlp = nn.Sequential(
            nn.Linear(allele_embed_dim + 1, allele_embed_dim),  # +1 for frequency
            nn.ReLU(),
            nn.Linear(allele_embed_dim, allele_embed_dim),
            nn.ReLU()
        )

        # -------------------------------------------------
        # 2️⃣ Marker-level MLP (after comparing episodes)
        # -------------------------------------------------
        self.marker_mlp = nn.Sequential(
            nn.Linear(allele_embed_dim * 4, marker_embed_dim),
            nn.ReLU(),
            nn.Linear(marker_embed_dim, marker_embed_dim),
            nn.ReLU()
        )

        # -------------------------------------------------
        # 3️⃣ Pair-level MLP (aggregate markers + priors + MOI)
        # -------------------------------------------------
        self.pair_mlp = nn.Sequential(
            nn.Linear(marker_embed_dim + 3 + 2, pair_embed_dim),  # +3 priors +2 MOI
            nn.ReLU(),
            nn.Linear(pair_embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, X_alleles, allele_mask, marker_mask, priors, MOI):
        """
        X_alleles : (batch, M_max, 2, A_max, 2)
        allele_mask: same shape as X_alleles[:,:,:,:,0]
        marker_mask: (batch, M_max)
        priors     : (batch, 3)
        MOI        : (batch, 2)
        """
        B, M, E, A, _ = X_alleles.shape

        # ---------------------------------------------
        # 1️⃣ Embed alleles
        # ---------------------------------------------
        allele_ids = X_alleles[..., 0].long()
        freqs = X_alleles[..., 1]

        allele_emb = self.allele_embedding(allele_ids)  # (B,M,E,A,allele_embed_dim)
        allele_input = torch.cat([allele_emb, freqs.unsqueeze(-1)], dim=-1)
        allele_out = self.allele_mlp(allele_input)      # (B,M,E,A,allele_embed_dim)

        # Apply allele mask
        allele_out = allele_out * allele_mask.unsqueeze(-1)

        # ---------------------------------------------
        # 2️⃣ Aggregate alleles → marker embedding per episode
        # ---------------------------------------------
        # Sum pooling over alleles
        marker_ep_emb = allele_out.sum(dim=3)  # (B,M,E,allele_embed_dim)

        # ---------------------------------------------
        # 3️⃣ Compare episodes (ep1 vs ep2)
        # ---------------------------------------------
        ep1 = marker_ep_emb[:, :, 0, :]
        ep2 = marker_ep_emb[:, :, 1, :]

        marker_pair = torch.cat([ep1, ep2, torch.abs(ep1 - ep2), ep1 * ep2], dim=-1)  # (B,M,4*allele_embed_dim)

        # Apply marker-level MLP
        marker_emb = self.marker_mlp(marker_pair)  # (B,M,marker_embed_dim)

        # Masked sum over markers
        marker_emb = marker_emb * marker_mask.unsqueeze(-1)
        pair_emb = marker_emb.sum(dim=1)  # (B, marker_embed_dim)

        # ---------------------------------------------
        # 4️⃣ Concatenate priors + MOI
        # ---------------------------------------------
        pair_emb = torch.cat([pair_emb, priors, MOI], dim=-1)  # (B, marker_dim+4)

        # ---------------------------------------------
        # 5️⃣ Pair-level MLP → logits
        # ---------------------------------------------
        logits = self.pair_mlp(pair_emb)  # (B, n_classes)
        probs = F.softmax(logits, dim=-1)

        return probs


In [11]:
import json
# reading allele dictionary
with open('allele_dict.json', 'r') as file:
    allele_to_id = json.load(file)

In [12]:
# 1. Define the device
device = torch.device("cpu")

# 2. Initialize the model with the same parameters used during training
# Make sure len(allele_to_id) matches the training setup!
n_alleles = len(allele_to_id) 
model = PvDeepSets(n_alleles=n_alleles)
print(n_alleles)
# 3. Load the weights (state_dict)
# Replace "pv_deepsets_weights.pth" with your actual filename
model.load_state_dict(torch.load("../src/DeepSets/pv_deepsets_weights20260402.pth", map_location=device))



143


C:\Users\sinel\AppData\Local\Temp\ipykernel_38672\1365319441.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../src/DeepSets/pv_deepse

<All keys matched successfully>

In [14]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

def get_model_results(model, dataloader, device, threshold=0.5):
    model.eval()
    all_predicted = []
    y_pred_filtered = []
    undetermined_count = 0

    with torch.no_grad():
        for X_alleles, allele_mask, marker_mask, priors, MOI in dataloader:
            # ... (your existing GPU transfer code) ...
            y_pred = model(X_alleles, allele_mask, marker_mask, priors, MOI)
            
            # Apply Softmax if your model doesn't already output probabilities
            # probs = torch.softmax(y_pred, dim=1) 
            probs = y_pred 

            # Get max probabilities and their indices
            max_probs, preds = torch.max(probs, dim=1)

            for i in range(len(max_probs)):
                if max_probs[i] >= threshold:
                    y_pred_filtered.append(preds[i].item())
                else:
                    undetermined_count += 1

            all_predicted.append(y_pred.cpu().numpy())

    print(f"Total undetermined samples removed: {undetermined_count}")
    return np.vstack(all_predicted), y_pred_filtered

predicted_probs, y_pred_hard = get_model_results(model, test_loader, device)




Total undetermined samples removed: 1


In [17]:
predicted_probs[1].sum()

np.float32(1.0)

In [ ]:
# Step 1: removing patients with only one episode
# Step 2: make all possible pair (including making episode 1&2)
# step 3: Calculate MOIs
# step 4: Calculate Afreq